[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/05_machine_learning_fundamentals/05_learning_curves_and_error_analysis.ipynb)

# Bài 05 — Learning curves, chẩn đoán model và error analysis

> Tương ứng các bài *"Chẩn đoán Bias vs. Variance & Regularization"*, *"Learning Curves"* và *"Machine Learning System Design"* trong kế hoạch Tuần 6 của khoá AIVN.

**🎯 Mục tiêu**
- Vẽ và đọc **learning curve** để biết model đang bị high bias hay high variance.
- Quyết định **nên làm gì tiếp theo** (thêm dữ liệu? thêm feature? đổi λ?) dựa trên chẩn đoán — thay vì thử mò.
- Dùng **mức hiệu năng của con người / baseline** để tách *avoidable bias* và *variance*.
- **Error analysis**: phân loại lỗi để ưu tiên công việc — kỹ năng quan trọng nhất khi thiết kế hệ thống ML.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/05_machine_learning_fundamentals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from ml_course.datasets import load_digits_data, make_classification_2d
from ml_course.visualization import plot_learning_curve, set_style

set_style()

## 1. Learning curve là gì?

**Learning curve** vẽ lỗi (hoặc điểm số) trên **train** và **validation** theo **số lượng mẫu train** $m$:
- $m$ nhỏ: model dễ khớp hoàn hảo vài mẫu → lỗi train thấp, lỗi validation cao.
- $m$ tăng: lỗi train tăng dần (khó khớp hết), lỗi validation giảm dần (học được quy luật tổng quát hơn).

Hình dạng hai đường khi $m$ lớn cho biết vấn đề của model:

In [ ]:
X, y = make_classification_2d("moons", n=2000, noise=0.25, seed=0)
sizes = np.linspace(0.05, 1.0, 10)
models = {
    "Logistic Regression tuyến tính — HIGH BIAS": make_pipeline(StandardScaler(), LogisticRegression()),
    "Decision Tree không giới hạn — HIGH VARIANCE": DecisionTreeClassifier(random_state=0),
    "Logistic + đặc trưng đa thức bậc 3 — VỪA": make_pipeline(PolynomialFeatures(3), StandardScaler(), LogisticRegression(max_iter=2000)),
}
fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
for ax, (name, model) in zip(axes, models.items()):
    n_train, train_scores, val_scores = learning_curve(model, X, y, train_sizes=sizes, cv=5, scoring="accuracy")
    plot_learning_curve(n_train, 1 - train_scores.mean(1), 1 - val_scores.mean(1), ax=ax, metric="tỉ lệ lỗi", title=name)
    ax.set_ylim(0, 0.25)
plt.show()

| Hình dạng | Chẩn đoán | Thêm dữ liệu có giúp? |
|---|---|---|
| Hai đường **hội tụ** ở mức lỗi **cao** | **high bias** — model quá đơn giản | ❌ Không (đường đã phẳng) |
| Khoảng **cách lớn** giữa train (thấp) và validation (cao), validation **vẫn đang giảm** | **high variance** | ✅ Có |
| Hội tụ ở mức lỗi **thấp**, gần mức mong muốn | tốt | — |

## 2. "Nên làm gì tiếp theo?" — bảng quyết định

Khi model chưa đủ tốt, có rất nhiều thứ có thể thử. Chẩn đoán trước giúp tránh lãng phí hàng tuần (vd: thu thập thêm dữ liệu cho một model đang high bias là vô ích).

| Hành động | Sửa vấn đề |
|---|---|
| Thu thập thêm dữ liệu train | high variance |
| Dùng ít feature hơn / feature selection | high variance |
| Tăng regularization λ | high variance |
| Ensemble (bagging), data augmentation, early stopping | high variance |
| Thêm feature / feature đa thức / tương tác | high bias |
| Giảm regularization λ | high bias |
| Model mạnh hơn (nhiều tham số hơn, cây sâu hơn, mạng lớn hơn) | high bias |
| Train lâu hơn / optimizer tốt hơn | high bias (do tối ưu chưa xong) |

## 3. Bias so với mốc tham chiếu: avoidable bias

"Lỗi cao" là cao **so với cái gì**? Dùng một **mốc tham chiếu** — lỗi của con người chuyên gia, hệ thống hiện tại, hoặc giới hạn lý thuyết (Bayes error):

```text
Mốc tham chiếu (vd: con người)   2%  ┐ avoidable bias = 8%   ← tập trung giảm bias
Lỗi train                       10%  ┘
Lỗi validation                  11%  ] variance = 1%
```
```text
Mốc tham chiếu                   2%  ] avoidable bias = 1%
Lỗi train                        3%  ┐ variance = 9%         ← tập trung giảm variance
Lỗi validation                  12%  ┘
```

Cùng lỗi validation ~11–12% nhưng chiến lược khắc phục hoàn toàn khác nhau.

## 4. Error analysis — nhìn vào các lỗi cụ thể

Andrew Ng: *"Đừng đoán — hãy nhìn vào dữ liệu."* Quy trình:
1. Lấy một mẫu (~100) các ví dụ model **dự đoán sai** trên validation.
2. Phân loại chúng theo nguyên nhân (đặt tên cho từng nhóm lỗi).
3. Đếm tỉ lệ mỗi nhóm → biết **trần cải thiện** nếu sửa nhóm đó.
4. Ưu tiên nhóm có tỉ lệ lớn và dễ sửa.

Ví dụ: phân loại chữ số viết tay 0–9 (bộ digits) bằng Logistic Regression.

In [ ]:
X_d, y_d, images = load_digits_data()
idx = np.arange(len(y_d))
X_tr, X_val, y_tr, y_val, idx_tr, idx_val = train_test_split(X_d, y_d, idx, test_size=0.3, random_state=0, stratify=y_d)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1)).fit(X_tr, y_tr)
pred = clf.predict(X_val)
wrong = np.flatnonzero(pred != y_val)
print(f"Accuracy validation: {np.mean(pred == y_val):.3f} — {len(wrong)} lỗi / {len(y_val)}")

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3.6))
for ax, i in zip(axes.ravel(), wrong[:20]):
    ax.imshow(images[idx_val[i]], cmap="gray_r")
    ax.set_title(f"thật {y_val[i]} → đoán {pred[i]}", fontsize=8)
    ax.axis("off")
fig.suptitle("Một số ví dụ bị dự đoán sai — chúng có điểm gì chung?")
plt.show()

Đếm các **cặp nhầm lẫn** phổ biến nhất (bản thu gọn của confusion matrix — chương 14):

In [ ]:
pairs = pd.Series([f"{t}→{p}" for t, p in zip(y_val[wrong], pred[wrong])]).value_counts()
print(pairs.head(8))
print(f"\n3 cặp nhầm lẫn hàng đầu chiếm {pairs.head(3).sum() / len(wrong):.0%} tổng số lỗi")

Sau khi xem ảnh, ta đặt tên cho các nhóm lỗi, ví dụ:

| Nhóm lỗi | % lỗi (ước tính khi xem ảnh) | Hướng xử lý |
|---|---|---|
| Nhầm giữa các chữ số có nét giống nhau (1↔8, 3↔8, 9↔5...) | phần lớn | feature tốt hơn / model phi tuyến (KNN, SVM-RBF, CNN) |
| Chữ viết lệch, nghiêng, đậm/mảnh bất thường | một phần | data augmentation (xoay, dịch), CNN |
| Ảnh mơ hồ, chính con người cũng khó đoán | ít | gần như không thể sửa — thuộc "Bayes error" |
| Nhãn sai | ít | sửa nhãn dữ liệu |

> Nếu nhóm "ảnh mơ hồ" chiếm 50% lỗi, thì dù cải thiện mọi thứ khác hoàn hảo, ta chỉ giảm được tối đa một nửa lỗi → ước lượng được **trần** của mỗi hướng cải thiện **trước khi** bỏ công.

## 5. Nguyên tắc thiết kế hệ thống ML

1. **Bắt đầu đơn giản, lặp nhanh:** có một pipeline end-to-end với model đơn giản (baseline) trong vài giờ/ngày, rồi cải thiện dựa trên chẩn đoán.
2. **Một con số đánh giá duy nhất** (single-number metric) + các ràng buộc (latency, kích thước model) để so sánh nhanh các thử nghiệm.
3. **Validation/test giống dữ liệu thật** khi triển khai (bài 08).
4. **Chẩn đoán trước khi hành động:** learning curve, bias/variance, error analysis.
5. **Dữ liệu lệch lớp (skewed classes):** accuracy gây hiểu lầm — 99% giao dịch hợp lệ thì model "luôn đoán hợp lệ" đạt 99% accuracy nhưng vô dụng. Dùng precision/recall/F1 (chương 14, 19).

In [ ]:
y_skewed = (np.random.default_rng(0).random(10_000) < 0.01).astype(int)   # 1% gian lận
always_zero = np.zeros_like(y_skewed)
print(f"Model 'luôn đoán hợp lệ': accuracy = {np.mean(always_zero == y_skewed):.3f}, "
      f"recall gian lận = {np.sum((always_zero == 1) & (y_skewed == 1)) / y_skewed.sum():.0%}")

## 🧠 Kiểm tra nhanh

1. Learning curve: lỗi train và validation hội tụ ở 25% khi con người đạt 5%. Có nên thu thập thêm dữ liệu không? Nên làm gì?
2. Lỗi train 1%, validation 15%, con người 1%. Chẩn đoán và 3 hành động phù hợp.
3. Trong error analysis, vì sao cần đếm tỉ lệ từng nhóm lỗi thay vì sửa ngay lỗi đầu tiên thấy được?

## 📌 Tóm tắt

- Learning curve: hội tụ ở lỗi cao → high bias; khoảng cách lớn và còn giảm → high variance.
- Bảng "nên làm gì tiếp theo" ánh xạ chẩn đoán → hành động.
- Avoidable bias = lỗi train − mốc tham chiếu; variance = lỗi validation − lỗi train.
- Error analysis: nhìn lỗi thật, phân nhóm, đếm, ưu tiên theo trần cải thiện.

## 📚 Tài liệu tham khảo

- Andrew Ng — *Machine Learning Yearning* (miễn phí), chương 13–32
- Andrew Ng — Coursera Machine Learning, tuần 6: *Advice for Applying ML* và *ML System Design*